In [1]:
import re
import pandas as pd

IN_PATH  = "autism_ai_screening_sheet_ALL168.xlsx"
OUT_PATH = "duplicates_report_by_doi.xlsx"

# -----------------------------
# Helpers
# -----------------------------
def normalize_doi(doi: str) -> str:
    """
    Normalize DOI strings to improve matching across sources:
    - lowercase
    - strip whitespace
    - remove leading URL forms like https://doi.org/
    - remove trailing punctuation
    """
    if not isinstance(doi, str):
        return ""
    doi = doi.strip().lower()
    doi = re.sub(r"^https?://(dx\.)?doi\.org/", "", doi)
    doi = doi.rstrip(" .;,)")
    return doi

def normalize_title(title: str) -> str:
    """
    Optional fallback: normalize title for matching when DOI is missing.
    """
    if not isinstance(title, str):
        return ""
    t = title.lower()
    t = re.sub(r"[^a-z0-9]+", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

# -----------------------------
# Load screening sheet
# -----------------------------
df = pd.read_excel(IN_PATH, sheet_name="Screening")

# Ensure columns exist
for col in ["DOI", "PMID", "Title"]:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# Normalize identifiers
df["DOI_norm"] = df["DOI"].apply(normalize_doi)
df["PMID_norm"] = df["PMID"].astype(str).replace("nan", "").str.strip()
df["Title_norm"] = df["Title"].apply(normalize_title)

# -----------------------------
# Define duplicate key
# Priority: DOI > PMID > Title (only if DOI+PMID missing)
# -----------------------------
def build_dup_key(row) -> str:
    if row["DOI_norm"]:
        return "DOI:" + row["DOI_norm"]
    if row["PMID_norm"] and row["PMID_norm"] != "0":
        return "PMID:" + row["PMID_norm"]
    # optional fallback — remove if you want DOI-only duplicates
    return "TITLE:" + row["Title_norm"]

df["DupKey"] = df.apply(build_dup_key, axis=1)

# Mark duplicates (anything sharing a DupKey with another row)
df["IsDuplicate"] = df.duplicated("DupKey", keep=False)

# Create a group ID for duplicates
dup_groups = df[df["IsDuplicate"]].copy()
dup_groups["DuplicateGroupID"] = dup_groups.groupby("DupKey").ngroup().apply(lambda x: f"D{str(x+1).zfill(4)}")

# Count duplicates per key
dup_counts = (
    dup_groups.groupby("DupKey")
    .size()
    .reset_index(name="Count")
    .sort_values("Count", ascending=False)
)

# Add count into the dup_groups table
dup_groups = dup_groups.merge(dup_counts, on="DupKey", how="left")

# Sort for readability
dup_groups = dup_groups.sort_values(["Count", "DupKey", "Database", "Record ID"], ascending=[False, True, True, True])

# -----------------------------
# Save outputs
# -----------------------------
with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as writer:
    # Full sheet with duplicate flag
    df.to_excel(writer, index=False, sheet_name="AllRecords_WithFlags")

    # Only duplicates, grouped
    dup_groups.to_excel(writer, index=False, sheet_name="Duplicates_Grouped")

    # Summary of duplicate keys
    dup_counts.to_excel(writer, index=False, sheet_name="DuplicateKey_Summary")

print("Saved duplicate report to:", OUT_PATH)

Saved duplicate report to: duplicates_report_by_doi.xlsx


In [2]:
import pandas as pd

df = pd.read_excel("autism_ai_screening_sheet_ALL168.xlsx", sheet_name="Screening")

# Count True values
n_true = df["IsDuplicate"].sum()

print("Number of True values:", n_true)

KeyError: 'IsDuplicate'